# Probability distributions
> Sample and evaluate probability distributions

In [ ]:
]help •normal

A distribution constructor, such as `•normal 0 1`, returns a keyed vector of four functions:

- `sample shape` draws random values. `g sample shape` draws them from a generator made by `•rand`.
- `density x` gives the probability density, or the probability mass for a discrete distribution.
- `cdf x` gives P(X ≤ x).
- `quantile p` inverts the CDF.

Parameters are finite real scalars or vectors. Scale, shape, rate and degrees of freedom are positive, except where stated.

| Constructor | Parameters |
|---|---|
| `•normal` | μ σ: mean, standard deviation |
| `•uniform` | a b: lower and upper bounds, a < b |
| `•bernoulli` | p ∈ [0,1] |
| `•binomial` | n p: integer trials n ≥ 0, p ∈ [0,1] |
| `•poisson` | λ ≥ 0: mean |
| `•beta` | α β: shapes |
| `•gamma` | k θ: shape, scale, with mean kθ |
| `•inversegamma` | α β: shape, scale, with density ∝ x⁻⁽ᵅ⁺¹⁾ exp(−β/x) |
| `•exponential` | λ: rate, with mean 1/λ |
| `•chisquared` | ν: degrees of freedom |
| `•student` | ν: degrees of freedom, with location 0 and scale 1 |
| `•fisher` | ν₁ ν₂: degrees of freedom |
| `•cauchy`, `•laplace`, `•logistic` | location, scale |
| `•lognormal` | μ σ: mean and standard deviation of log(X) |
| `•weibull` | k λ: shape, scale |

Errors: DOMAIN for invalid parameters, non-real inputs or p ∉ [0,1]; LENGTH for the wrong number of parameters; RANK for matrix parameters or shapes; SYNTAX for dyadic calls other than `sample`; LIMIT for oversized shapes or sampler ranges.

Construct a distribution, then sample or evaluate it:

In [ ]:
n←•normal 0 1
⍴n.sample 2 3
n.cdf 0
n.quantile 0.5
n.density 0

2ₓ 3ₓ

0.5

0

0.39894228040143265

`sample 100` returns a vector. `sample ⍬` returns a rank-0 array. Zero dimensions give empty arrays. Continuous samples are floats. Discrete samples are exact integers:

In [ ]:
b←•binomial 2 0.5
b.density 0 1 2
b.cdf 0.5 1.5
b.quantile 0.25 0.5 1

0.25 0.5 0.25

0.25000000000000094 0.7499999999999991

0ₓ 1ₓ 2ₓ

Evaluation pervades arrays, preserving shape, nesting, keys and axis names. Parameters belong to the constructed distribution.

In [ ]:
u←•uniform 0 1
u.cdf ("low" "high":0.2 0.8)

("low":0.2 ⋄ "high":0.8)

## Reproducible draws

`•rand seed` makes a generator. Put it on the left of `sample` to draw from a seeded stream. The same seed gives the same draws. Saved output then stays the same when a notebook runs again. A generator also has `roll` and `deal`, which work like `?`. Copies of a generator share one stream.

In [ ]:
]help •rand

`•rand seed` returns a generator: a keyed vector of two functions that draw from one stream of random numbers. The seed is a nonnegative integer. The same seed gives the same draws.

- `roll Y` works like `?Y`.
- `X deal Y` works like `X?Y`.

A distribution's `sample` takes a generator on its left, as in `g d.sample 3`. Copies of a generator share its stream. Drawing from one copy moves every copy on.

Errors: DOMAIN for a seed that is not a nonnegative integer, or a left argument to `sample` that is not a generator; LENGTH or RANK for more than one seed; SYNTAX for a dyadic call to `•rand`.

In [ ]:
g←•rand 42
g n.sample 3
g.roll 6 6 6
3 g.deal 10

0.8343975468437962 ¯0.514962928147295 1.4077275731197503

5 5 4

2 6 3

## Parameters and support

Gamma takes scale, whereas exponential takes rate. These describe the same distribution:

In [ ]:
g←•gamma 1 2
e←•exponential 0.5
(g.cdf 2) = e.cdf 2

1ₓ

Quantile endpoints give the support bounds, including infinity. Discrete quantiles return the smallest supported integer whose CDF reaches p. p=0 gives the lower support bound. Certain events stay constant at both endpoints:

In [ ]:
n←•normal 0 1
n.quantile 0 1
p←•poisson 0
p.sample 3

¯∞ ∞

0ₓ 0ₓ 0ₓ

## Numerics

Numerical routines use [statrs](https://docs.rs/statrs/latest/statrs/). Logistic uses its closed-form CDF and inverse. Unseeded draws use the thread-local RNG. Draws from a `•rand` generator use Xoshiro256++. Binomial trials and Poisson rate are limited to 2⁵³ by the floating-point samplers. Gamma scale must have a finite reciprocal. Uniform intervals must fit the sampler's finite range.